# Week 7 Assignment — Building a Tool-Using Agent with LangChain

**Module 7: Agentic Tools in Python**

**Name:** Nagarajan Kanagamani

## Objective

Build a small LangChain agent that can decide, on its own, whether a question needs a **calculator**, a **web search**, or both — then call the right tool and return a clean final answer.

This notebook follows the five steps of the assignment:

1. Environment Setup
2. Initialise the LLM
3. Load Tools (calculator + search)
4. Build the Agent
5. Run and Observe

> **Note on API keys:** This notebook reads all keys from a `.env` file (see the `env_example` file already in this course folder) via `python-dotenv`. Never hard-code API keys directly in a notebook cell.

## Step 1 — Environment Setup

Make sure you have **Python 3.9+**, then install the required libraries:

```
pip install langchain langchain-openai langchain-community python-dotenv google-search-results duckduckgo-search
```

* `langchain` / `langchain-community` — agent framework, tool loaders (`llm-math`, `serpapi`)
* `langchain-openai` — `ChatOpenAI` chat model wrapper
* `google-search-results` — SerpAPI client, used if you have a `SERPAPI_API_KEY`
* `duckduckgo-search` — a free, no-key search tool used automatically as a fallback if SerpAPI isn't configured
* `python-dotenv` — loads `OPENAI_API_KEY` / `SERPAPI_API_KEY` from a local `.env` file instead of hard-coding them

Run the cell below once to install everything (safe to re-run; already-installed packages are skipped).

In [ ]:
# Run once. Comment out after the first successful install to save time on reruns.
%pip install -q langchain langchain-openai langchain-community python-dotenv google-search-results duckduckgo-search

In [ ]:
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 9), "This assignment requires Python 3.9 or newer."


In [ ]:
import os
from dotenv import load_dotenv

# Loads variables from a local .env file (copy env_example -> .env and fill in your keys).
load_dotenv()

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")   # course default provider, OpenAI-compatible endpoint
SERPAPI_API_KEY  = os.getenv("SERPAPI_API_KEY")

if not (OPENAI_API_KEY or TOGETHER_API_KEY):
    raise EnvironmentError(
        "No LLM key found. Set OPENAI_API_KEY (for OpenAI) or TOGETHER_API_KEY "
        "(for Together's OpenAI-compatible endpoint) in your .env file."
    )

print("OPENAI_API_KEY  :", "found" if OPENAI_API_KEY else "not set")
print("TOGETHER_API_KEY:", "found" if TOGETHER_API_KEY else "not set")
print("SERPAPI_API_KEY :", "found" if SERPAPI_API_KEY else "not set (will fall back to DuckDuckGo search)")


## Step 2 — Initialise Your LLM

We use `ChatOpenAI` from `langchain-openai`. Temperature is kept **low (0.2)** so the
model behaves predictably and reliably follows the tool-calling format the agent needs —
high temperature makes an agent more likely to hallucinate tool names or skip calling a
tool it actually needs.

If `OPENAI_API_KEY` is set we talk to OpenAI directly. Otherwise we point `ChatOpenAI` at
Together AI's OpenAI-compatible endpoint using `TOGETHER_API_KEY`, which is the course's
default provider (see `env_example`).

In [ ]:
from langchain_openai import ChatOpenAI

if OPENAI_API_KEY:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_OPENAI", "gpt-4o-mini"),
        temperature=0.2,
        api_key=OPENAI_API_KEY,
    )
    print("Using OpenAI as the LLM provider.")
else:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_DEFAULT", "meta-llama/Llama-3.3-70B-Instruct-Turbo"),
        temperature=0.2,
        api_key=TOGETHER_API_KEY,
        base_url="https://api.together.xyz/v1",
    )
    print("Using Together AI as the LLM provider.")

# Quick sanity check that the LLM responds at all before we wire it into an agent.
llm.invoke("Reply with the single word: ready").content


## Step 3 — Load Tools

We register **two tools**:

1. **Calculator** — LangChain's built-in `llm-math` tool, backed by an `LLMMathChain`.
2. **Search** — `serpapi` if `SERPAPI_API_KEY` is set; otherwise a free `DuckDuckGoSearchRun`
   tool, wrapped so the agent sees a tool literally named `Search` either way.

Each tool is invoked **independently** first, so we can confirm it works in isolation
before handing both to an agent.

In [ ]:
from langchain.agents import load_tools, Tool

# --- Calculator tool ---
math_tool = load_tools(["llm-math"], llm=llm)[0]

# --- Search tool: SerpAPI if available, else DuckDuckGo (no key required) ---
if SERPAPI_API_KEY:
    search_tool = load_tools(["serpapi"], serpapi_api_key=SERPAPI_API_KEY)[0]
    search_tool.name = "Search"
    print("Search tool: SerpAPI")
else:
    from langchain_community.tools import DuckDuckGoSearchRun
    ddg = DuckDuckGoSearchRun()
    search_tool = Tool(
        name="Search",
        func=ddg.run,
        description="Useful for answering questions about current events, facts, "
                     "or anything you don't already know. Input should be a search query.",
    )
    print("Search tool: DuckDuckGo (fallback, no API key needed)")

tools = [math_tool, search_tool]
[t.name for t in tools]


In [ ]:
# Confirm each tool can be invoked independently before building the agent.
print("Calculator test  ->", math_tool.run("What is 47 * 89 + 12?"))
print()
print("Search test      ->", search_tool.run("current population of India")[:300], "...")


## Step 4 — Build the Agent

We combine the chat model and the two tools with `initialize_agent`, using the
`ZERO_SHOT_REACT_DESCRIPTION` agent type (a ReAct-style agent that decides, at each
step, whether to call a tool or give a final answer, based on the tool descriptions
above).

A short system prompt is injected through `agent_kwargs["prefix"]`, telling the model to:

* call a tool whenever the question needs a calculation or a fact it isn't sure of,
* never guess at arithmetic or current facts on its own,
* and always return a clean, structured final answer.

`verbose=True` prints the agent's intermediate `Thought / Action / Action Input /
Observation` trace, which we use in Step 5 to inspect its reasoning.

In [ ]:
from langchain.agents import initialize_agent, AgentType

SYSTEM_PREFIX = (
    "You are a careful, tool-using assistant. "
    "You have access to a Calculator tool and a Search tool. "
    "Always call a tool when the question involves arithmetic or a fact you are not "
    "certain of — never guess or compute by hand. "
    "Once you have enough information, respond with a concise, clearly formatted final answer, "
    "showing the key numbers or facts you used."
)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    agent_kwargs={"prefix": SYSTEM_PREFIX},
)
agent


## Step 5 — Run and Observe

We issue three kinds of queries and inspect the printed reasoning trace above each result:

1. A **pure math** query — should trigger only the Calculator tool.
2. A **pure search** query — should trigger only the Search tool.
3. A **combined** query (the assignment's example: *"How many hours in X days?"*, extended
   to also require a looked-up fact) — should trigger both tools in sequence.

Read the `Thought / Action / Action Input / Observation` trace for each run and note,
in the Observations section below, whenever the agent picks the wrong tool, skips a
tool it needed, or otherwise goes off track.

In [ ]:
# 1) Pure math query
result_math = agent.invoke({"input": "What is 47 times 89, plus 12?"})
print("\nFINAL ANSWER:", result_math["output"])


In [ ]:
# 2) Pure search query
result_search = agent.invoke({"input": "Who is the current Prime Minister of India?"})
print("\nFINAL ANSWER:", result_search["output"])


In [ ]:
# 3) Combined math + search query (course example: "How many hours in X days?")
result_combo = agent.invoke({
    "input": "How many hours are there in 45 days? Also, tell me one interesting fact "
             "about the number 45."
})
print("\nFINAL ANSWER:", result_combo["output"])


In [ ]:
# Bonus: a query designed to need BOTH tools in one chain of reasoning
result_bonus = agent.invoke({
    "input": "Search for the current population of India, then calculate what that "
             "number would be if it grew by 5 percent."
})
print("\nFINAL ANSWER:", result_bonus["output"])


## Observations & Reflections

Fill this section in after reading the verbose traces above.

| # | Query | Tool(s) expected | Tool(s) actually called | Correct? | Notes |
|---|-------|-------------------|--------------------------|----------|-------|
| 1 | Pure math | Calculator | | | |
| 2 | Pure search | Search | | | |
| 3 | Combined (hours + fact) | Calculator, Search | | | |
| 4 | Bonus (search then calculate) | Search, Calculator | | | |

**Did the agent ever choose the "wrong" tool, or fail to call a tool when it should have?**

_(e.g. tried to compute the multiplication itself instead of calling Calculator; called
Search for something it should have just calculated; stopped after one tool when the
question needed both)_

- ...

**What prompt or configuration changes fixed it?**

_(e.g. making the system prefix more explicit about always using tools for arithmetic;
lowering temperature further; rewording a tool's `description` so the agent picks it up
in the right situations; adding `handle_parsing_errors=True` to recover from malformed
tool calls)_

- ...

**Key takeaway**

_(one or two sentences on what you learned about how a ReAct-style agent decides which
tool to call, based only on the tool names/descriptions and the system prompt)_

- ...